# Solutions · 13 · Discrete-event simulation

Worked solutions to the exercises of [notebook 13](../notebooks/13_discrete_event_simulation.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engstoch is missing): install it from GitHub
    import engstoch
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engstoch"], check=True)
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from collections import deque
from engstoch import des, montecarlo as mc
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

R = np.random.default_rng      # R(seed): a fresh, reproducible generator

## Exercise 1

Add a callback option to the call-centre model: callers who abandon call back after an exponential time (mean 30 min) with probability 0.5. How much does it raise the load and the abandonment rate at 4 agents?

In [2]:
def centre_with_callbacks(c, retry_prob, seed, T=20_000.0, lam=4.0, mu=1.0, theta=0.5, retry_mean=0.5):
    g = R(seed); sim = des.Simulator(); queue = deque(); busy = [0]; stats_ = {"fresh": 0, "retries": 0, "abandon": 0, "served": 0}
    def arrive(kind):
        stats_[kind] += 1
        call = {"start": None}
        if busy[0] < c and not queue: start(call)
        else:
            queue.append(call); sim.after(g.exponential(1 / theta), abandon, call)
    def fresh():
        arrive("fresh"); sim.after(g.exponential(1 / lam), fresh)
    def start(call):
        call["start"] = sim.now; busy[0] += 1; sim.after(g.exponential(1 / mu), done)
    def done():
        busy[0] -= 1; stats_["served"] += 1
        if queue: start(queue.popleft())
    def abandon(call):
        if call["start"] is None and call in queue:
            queue.remove(call); stats_["abandon"] += 1
            if g.random() < retry_prob: sim.after(g.exponential(retry_mean), arrive, "retries")
    sim.schedule(0.0, fresh); sim.run(until=T)
    return stats_
for p in (0.0, 0.5):
    s = centre_with_callbacks(4, p, 1)
    tot = s["fresh"] + s["retries"]
    print(f"retry probability {p}: offered calls per fresh call {tot / s['fresh']:.3f}, abandonment per attempt {s['abandon'] / tot:.3f}, fresh callers never served {1 - s['served'] / s['fresh']:.3f}")

retry probability 0.0: offered calls per fresh call 1.000, abandonment per attempt 0.180, fresh callers never served 0.180


retry probability 0.5: offered calls per fresh call 1.149, abandonment per attempt 0.261, fresh callers never served 0.150


Callbacks add load exactly when the system is congested, which lengthens queues and causes more abandonment
per attempt; but because half the abandoners try again, the fraction of *customers* never served falls. Which
number a manager watches (abandoned attempts or lost customers) changes the conclusion - and logs that do not
link retries to callers overstate demand.

## Exercise 2

Compare continuous review with periodic review (every 2 time units) in the (s, S) inventory model: optimise each and report the cost difference with a common-random-numbers interval.

In [3]:
def cost(s, S, seed, review=None):
    return des.inventory_sS(s, S, 2.0, lambda g: 1 + g.poisson(1.0), lambda g: g.uniform(1.0, 3.0), 2000.0, R(seed), review=review)["cost_rate"]
grid = [(s, S) for s in range(4, 22, 2) for S in range(s + 10, s + 45, 5)]
best_c = min(grid, key=lambda k: np.mean([cost(*k, seed) for seed in range(4)]))
best_p = min(grid, key=lambda k: np.mean([cost(*k, seed, 2.0) for seed in range(4)]))
d = [cost(*best_p, 500 + k, 2.0) - cost(*best_c, 500 + k) for k in range(20)]
r = mc.mean_ci(d, method="t")
print(f"continuous review best {best_c}, periodic review (every 2) best {best_p}")
print(f"extra cost of periodic review: {r['estimate']:.2f} per unit time, 95 % CI ({r['ci'][0]:.2f}, {r['ci'][1]:.2f})")

continuous review best (6, 31), periodic review (every 2) best (12, 32)
extra cost of periodic review: 1.25 per unit time, 95 % CI (1.02, 1.48)


Periodic review must protect against demand over the lead time *plus* the review period, so its best reorder
point is higher and its cost a little larger; the common-random-numbers interval shows the difference is real.
In practice the saving in monitoring effort may be worth it.

## Exercise 3

**Implement it yourself:** add a priority class to the M/M/1 example above (two customer types, non-preemptive priority) and check the mean waits against Cobham's formula.

In [4]:
def priority_mm1(lam1, lam2, mu, T, seed):
    g = R(seed); sim = des.Simulator(); queues = [deque(), deque()]; busy = [False]; waits = [[], []]
    def arrive(k, lam):
        if not busy[0]: start(k, sim.now)
        else: queues[k].append(sim.now)
        sim.after(g.exponential(1 / lam), arrive, k, lam)
    def start(k, t_arr):
        busy[0] = True; waits[k].append(sim.now - t_arr); sim.after(g.exponential(1 / mu), done)
    def done():
        busy[0] = False
        for k in (0, 1):
            if queues[k]: start(k, queues[k].popleft()); break
    sim.schedule(0.0, arrive, 0, lam1); sim.schedule(0.0, arrive, 1, lam2); sim.run(until=T)
    return [np.mean(w[1000:]) for w in waits]
lam1, lam2, mu = 0.3, 0.5, 1.0
w = priority_mm1(lam1, lam2, mu, 200_000.0, 1)
W0 = (lam1 + lam2) / mu**2                                   # sum lambda_i E[S_i^2]/2 with E[S^2] = 2/mu^2
s1, s = lam1 / mu, (lam1 + lam2) / mu
print(f"high priority: simulated {w[0]:.3f}, Cobham {W0 / (1 - s1):.3f}")
print(f"low priority:  simulated {w[1]:.3f}, Cobham {W0 / ((1 - s1) * (1 - s)):.3f}")

high priority: simulated 1.130, Cobham 1.143
low priority:  simulated 5.708, Cobham 5.714


Cobham's formula W_k = W₀ / ((1 − σ_{k−1})(1 − σ_k)) with W₀ = Σλᵢ E[Sᵢ²]/2 gives the mean wait of each class
under non-preemptive priority: about 1.1 for the high class and 5.7 for the low one, against 4.0 for everybody
under FCFS at the same total load 0.8. The load-weighted waits are conserved - 0.3 × 1.14 + 0.5 × 5.71 =
0.8 × 4.0 (Kleinrock's conservation law): priority moves waiting between classes, it does not remove it.